### Загрузка данных

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
df = pd.read_csv('../../data/raw/dataset.csv', index_col=0)
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
0,Алтайский край,2000,31.0,176660.0,46736.8,17660.5,122.0,52.8,569.0,109.2
1,Алтайский край,2001,34.0,111590.0,61854.4,23509.0,104.5,53.1,599.0,109.4
2,Алтайский край,2002,36.0,67466.0,73107.4,27991.2,103.1,53.2,563.0,100.1
3,Алтайский край,2003,36.0,50114.0,88733.3,34295.8,100.8,53.5,516.0,105.5
4,Алтайский край,2004,36.0,38270.0,114840.5,44934.9,99.5,53.7,488.0,102.6


In [3]:
df_features = pd.read_csv('../../data/features.csv')
df_features.head()

,Регион,Год,Заболеваемость_инфекции_на_1000,Коэффициент_смертности_населения,Численность_населения,Оборотная_вода_млн_м3
0,Белгородская область,2004,41.3,16.2,1511.7,1610.0
1,Брянская область,2004,33.8,19.1,1344.1,63.0
2,Владимирская область,2004,46.5,20.1,1497.6,339.0
3,Воронежская область,2004,26.1,18.5,2364.9,2419.0
4,Ивановская область,2004,31.6,21.6,1116.7,223.0


### Merge df_main + df_features

#### Проверка правильности регионов

In [7]:
def compare_unique(df1, df2):
    shared = set(df1.columns) & set(df2.columns)
    differences = {
        col: {
            'only_in_df1': sorted(set(df1[col].unique()) - set(df2[col].unique())),
            'only_in_df2': sorted(set(df2[col].unique()) - set(df1[col].unique()))
        }
        for col in shared
        if set(df1[col].unique()) != set(df2[col].unique())
    }
    identical = [col for col in shared if set(df1[col].unique()) == set(df2[col].unique())]
    if 'Регион' in identical:
        print("Различий уникальных значений по столбцу 'Регион' нет.")
    elif 'Регион' in differences:
        print("Есть различия по уникальным значениям в столбце 'Регион'.")
    return differences, identical


In [8]:
compare_unique(df_main, df_features)

Различий уникальных значений по столбцу 'Регион' нет.


({}, ['Год', 'Регион'])

#### Соединяем таблицы по двум ключам (inner join)

In [10]:
df_main.shape

(1992, 5)

In [11]:
df_features.shape

(1992, 6)

In [12]:
df = pd.merge(
    df_main, df_features,
    on=['Регион', 'Год'],
    how='inner',  
    suffixes=('_df1', '_df2')
)


In [13]:
df.rename(columns={
    'Объем_сточных_вод_млн_м3': 'V_сточ_вод_млн_м3',
    'Инвестиции_в_ООС': 'Инв_в_ООС',
    'ВРП': 'ВРП',
    'Заболеваемость_инфекции_на_1000': 'Забол_инф_на_1000',
    'Коэффициент_смертности_населения': 'Коэф_смерт_насел',
    'Численность_населения': 'Числ_насел',
    'Оборотная_вода_млн_м3': 'Оборот_вод_млн_м3'
}, inplace=True)

In [9]:
df.columns

Index(['Регион', 'Год', 'Объем_сточных_вод_млн_м3', 'Инвестиции_в_ООС_тыс_руб',
       'ВРП_млн_руб', 'ВРП_на_душу_руб', 'Индексы_производства_продукции_СХ_%',
       'Доля_городского_населения_%', 'Использование_свеж_воды_млн_м3',
       'Индекс_промыш_производства_%'],
      dtype='object')

In [46]:
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
0,Алтайский край,2000,31.0,176660.0,46736.8,17660.5,122.0,52.8,569.0,109.2
1,Алтайский край,2001,34.0,111590.0,61854.4,23509.0,104.5,53.1,599.0,109.4
2,Алтайский край,2002,36.0,67466.0,73107.4,27991.2,103.1,53.2,563.0,100.1
3,Алтайский край,2003,36.0,50114.0,88733.3,34295.8,100.8,53.5,516.0,105.5
4,Алтайский край,2004,36.0,38270.0,114840.5,44934.9,99.5,53.7,488.0,102.6


In [48]:
df.shape

(1992, 10)

In [50]:
df.isna().sum()

Регион                                  0
Год                                     0
Объем_сточных_вод_млн_м3                5
Инвестиции_в_ООС_тыс_руб                5
ВРП_млн_руб                             5
ВРП_на_душу_руб                        36
Индексы_производства_продукции_СХ_%    42
Доля_городского_населения_%             0
Использование_свеж_воды_млн_м3          0
Индекс_промыш_производства_%           21
dtype: int64

In [ ]:
# В стобце "Забол_инф_на_1000" много пропусков, удалим его
df = df.drop('Забол_инф_на_1000', axis=1)
df.head()

In [19]:
df.shape

(1992, 10)

#### Удаляем регион с пропусками

In [200]:
df[df['Объем_сточных_вод_млн_м3'] < 0.2]

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
756,Ненецкий автономный округ,2012,0.11,241307.0,157067.1000,3721264.5,106.6,69.9,28.97,89.3
757,Ненецкий автономный округ,2013,0.13,292813.0,173170.2000,4099188.4,81.5,70.7,33.48,96.5
758,Ненецкий автономный округ,2014,0.01,1180497.0,187009.8000,4426058.3,95.7,71.7,9.22,105.9
759,Ненецкий автономный округ,2015,0.01,1075468.0,227193.5000,5359728.6,110.6,72.3,14.09,110.2
763,Ненецкий автономный округ,2019,0.12,312068.0,330999.0000,7954412.5,95.3,73.4,15.58,98.9
764,Ненецкий автономный округ,2020,0.09,3650128.0,230674.3000,5547322.7,87.2,73.8,21.67,88.9
765,Ненецкий автономный округ,2021,0.00,2375.0,409388.3000,9866203.9,125.6,74.2,20.51,102.4
766,Ненецкий автономный округ,2022,0.00,300431.0,488014.4000,11786365.0,95.9,74.5,22.05,112.1
767,Ненецкий автономный округ,2023,0.00,515966.0,501455.4618,11995394.3,102.8,74.8,20.00,93.7


In [192]:
df[df['Инвестиции_в_ООС_тыс_руб'] == 0].shape[0]

84

In [202]:
df[df['Инвестиции_в_ООС_тыс_руб'] == 0].head(50)

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
127,Брянская область,2007,94.00,0.0,1.027062e+05,78518.8,106.0,68.8,123.00,121.8
129,Брянская область,2009,83.00,0.0,1.264774e+05,98014.5,103.4,69.0,110.00,76.6
130,Брянская область,2010,78.00,0.0,1.470240e+05,114777.6,94.2,69.1,117.00,122.9
131,Брянская область,2011,75.00,0.0,1.742118e+05,137227.5,124.6,69.2,111.00,110.6
135,Брянская область,2015,57.60,0.0,2.717825e+05,221557.6,113.1,69.5,96.29,113.5
136,Брянская область,2016,57.92,0.0,3.164894e+05,259301.1,108.9,69.6,95.13,107.3
137,Брянская область,2017,55.12,0.0,3.411778e+05,281250.4,106.0,69.7,93.95,104.1
138,Брянская область,2018,53.78,0.0,3.671571e+05,305284.8,103.1,69.7,90.02,102.7
139,Брянская область,2019,50.75,0.0,3.991138e+05,334495.9,100.4,69.6,89.58,117.0
140,Брянская область,2020,49.69,0.0,4.141794e+05,349774.4,103.0,69.5,86.92,102.0


In [5]:
df = df[df['Регион'] != 'Чеченская Республика']

#### Меняем формат числа у ВРП в 2023 году

In [8]:
df[df['Год'] == 2023] # сейчас данные находятся в тыс., а нужно в миллионах, как у остальных

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
23,Алтайский край,2023,13.0,2247651.0,1.024355e+06,482474.4,93.1,58.5,425.0,107.1
47,Амурская область,2023,60.0,2883951.0,7.938519e+05,1054056.2,94.2,68.5,115.0,97.2
71,Архангельская область,2023,247.0,2286387.0,7.615896e+05,793259.7,101.5,78.1,500.0,98.8
95,Астраханская область,2023,85.0,191976.0,7.777183e+05,819951.6,103.0,63.9,561.0,100.6
119,Белгородская область,2023,58.0,17411221.0,1.341409e+06,889768.6,105.0,65.3,224.0,104.7
...,...,...,...,...,...,...,...,...,...,...
1895,Чукотский автономный округ,2023,3.0,800015.0,1.867094e+05,3895053.5,91.5,69.4,100.0,110.0
1919,Ямало-Ненецкий автономный округ,2023,25.0,105498709.0,5.379402e+06,10462220.5,80.7,85.2,189.0,97.1
1943,Ярославская область,2023,143.0,9917079.0,8.497699e+05,713444.2,105.1,80.8,180.0,107.0
1967,г. Москва,2023,833.0,22791947.0,3.233900e+07,2463550.4,70.8,100.0,1331.0,119.0


In [10]:
df.loc[df['Год'] == 2023, 'ВРП_млн_руб'] = df.loc[df['Год'] == 2023, 'ВРП_млн_руб'] / 1

In [12]:
df['ВРП_млн_руб'] = df['ВРП_млн_руб'].round(1)

In [14]:
df[df['Год'] == 2023]

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
23,Алтайский край,2023,13.0,2247651.0,1024355.4,482474.4,93.1,58.5,425.0,107.1
47,Амурская область,2023,60.0,2883951.0,793851.9,1054056.2,94.2,68.5,115.0,97.2
71,Архангельская область,2023,247.0,2286387.0,761589.6,793259.7,101.5,78.1,500.0,98.8
95,Астраханская область,2023,85.0,191976.0,777718.3,819951.6,103.0,63.9,561.0,100.6
119,Белгородская область,2023,58.0,17411221.0,1341408.9,889768.6,105.0,65.3,224.0,104.7
...,...,...,...,...,...,...,...,...,...,...
1895,Чукотский автономный округ,2023,3.0,800015.0,186709.4,3895053.5,91.5,69.4,100.0,110.0
1919,Ямало-Ненецкий автономный округ,2023,25.0,105498709.0,5379401.8,10462220.5,80.7,85.2,189.0,97.1
1943,Ярославская область,2023,143.0,9917079.0,849769.9,713444.2,105.1,80.8,180.0,107.0
1967,г. Москва,2023,833.0,22791947.0,32339001.6,2463550.4,70.8,100.0,1331.0,119.0


#### Нормализуем Инвестии_в_ООС по ВРП

In [17]:
df['ВРП_млн_руб'] = df['ВРП_млн_руб'] * 1000

In [19]:
df['Инвестиции_норм'] = df['Инвестиции_в_ООС_тыс_руб'] / df['ВРП_млн_руб']

In [21]:
df = df.drop(['Инвестиции_в_ООС_тыс_руб', 'ВРП_млн_руб'], axis=1)

In [23]:
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Инвестиции_норм
0,Алтайский край,2000,31.0,17660.5,122.0,52.8,569.0,109.2,0.003780
1,Алтайский край,2001,34.0,23509.0,104.5,53.1,599.0,109.4,0.001804
2,Алтайский край,2002,36.0,27991.2,103.1,53.2,563.0,100.1,0.000923
3,Алтайский край,2003,36.0,34295.8,100.8,53.5,516.0,105.5,0.000565
4,Алтайский край,2004,36.0,44934.9,99.5,53.7,488.0,102.6,0.000333


### Сохранение датасета

In [25]:
df.describe()

,Год,Объем_сточных_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Инвестиции_норм
count,1992.000000,1987.000000,1.956000e+03,1950.000000,1992.000000,1992.000000,1971.000000,1987.000000
mean,2011.500000,188.086678,4.193183e+05,102.313692,69.739207,687.950271,105.041527,0.007459
std,6.923925,265.875971,8.532238e+05,10.775905,13.038158,936.328242,10.085978,0.011266
min,2000.000000,0.000000,6.667900e+03,41.400000,26.000000,4.720000,43.200000,0.000000
25%,2005.750000,38.030000,9.410377e+04,97.100000,63.400000,134.890000,100.610000,0.001582
50%,2011.500000,88.000000,2.253240e+05,101.600000,70.350000,305.000000,104.400000,0.004141
75%,2017.250000,212.000000,4.364428e+05,106.500000,77.700000,790.882500,109.000000,0.009077
max,2023.000000,2661.000000,1.199539e+07,185.000000,100.000000,6849.000000,273.700000,0.134296


In [27]:
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Инвестиции_норм
0,Алтайский край,2000,31.0,17660.5,122.0,52.8,569.0,109.2,0.003780
1,Алтайский край,2001,34.0,23509.0,104.5,53.1,599.0,109.4,0.001804
2,Алтайский край,2002,36.0,27991.2,103.1,53.2,563.0,100.1,0.000923
3,Алтайский край,2003,36.0,34295.8,100.8,53.5,516.0,105.5,0.000565
4,Алтайский край,2004,36.0,44934.9,99.5,53.7,488.0,102.6,0.000333


In [39]:
cols = list(df.columns)

In [25]:
df['Инвестиции_норм'] = df['Инвестиции_норм'] * 100

In [27]:
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Инвестиции_норм
0,Алтайский край,2000,31.0,17660.5,122.0,52.8,569.0,109.2,0.377989
1,Алтайский край,2001,34.0,23509.0,104.5,53.1,599.0,109.4,0.180408
2,Алтайский край,2002,36.0,27991.2,103.1,53.2,563.0,100.1,0.092283
3,Алтайский край,2003,36.0,34295.8,100.8,53.5,516.0,105.5,0.056477
4,Алтайский край,2004,36.0,44934.9,99.5,53.7,488.0,102.6,0.033324


In [29]:
df.rename(columns={
    'Инвестиции_норм': 'Инвестиции_норм_%',
}, inplace=True)

In [33]:
cols = list(df.columns)
cols

['Регион',
 'Год',
 'Объем_сточных_вод_млн_м3',
 'ВРП_на_душу_руб',
 'Индексы_производства_продукции_СХ_%',
 'Доля_городского_населения_%',
 'Использование_свеж_воды_млн_м3',
 'Индекс_промыш_производства_%',
 'Инвестиции_норм_%']

In [35]:
cols.remove('Инвестиции_норм_%')  

In [37]:
cols.insert(3, 'Инвестиции_норм_%') 

In [39]:
df = df[cols]   

In [41]:
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_норм_%,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
0,Алтайский край,2000,31.0,0.377989,17660.5,122.0,52.8,569.0,109.2
1,Алтайский край,2001,34.0,0.180408,23509.0,104.5,53.1,599.0,109.4
2,Алтайский край,2002,36.0,0.092283,27991.2,103.1,53.2,563.0,100.1
3,Алтайский край,2003,36.0,0.056477,34295.8,100.8,53.5,516.0,105.5
4,Алтайский край,2004,36.0,0.033324,44934.9,99.5,53.7,488.0,102.6


In [43]:
df.to_excel('../../data/dataset_11_11.xlsx')
df.to_csv('../../data/dataset_11_11.csv')